Reading CSV file from ADLS2

In [0]:
import datetime
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window


def getMountPointWithDate(container_name):
    currentdate = datetime.datetime.now()
    year = currentdate.year
    month = currentdate.month
    day = currentdate.day
    if container_name == "inboundinventory":
        return f"/mnt/{container_name}/I0132/{year}/{month:02d}/{day:02d}"
    else:
        return f"/mnt/{container_name}"

inbound_container = getMountPointWithDate("inboundinventory")
deltable_container = getMountPointWithDate("deltatablesinventory")
print('Mount point for raw inventory inbound layer',inbound_container)
print('Mount point for raw delta table',deltable_container)


In [0]:
shema_inbound_inventory_raw = StructType([
                                    #StructField("ID", LongType(), True),    
                                    StructField("ARTICLE_NUMBER", StringType(), True),
                                    StructField("WIN_NUMBER", IntegerType(), True),
                                    StructField("SITE", ShortType(), True),
                                    StructField("MERCHANDISE_CATEGORY", StringType(), True),
                                    StructField("REPORT_DATE", StringType(), True),
                                    StructField("BASE_UOM", StringType(), True),
                                    StructField("QUANTITY", DecimalType(9, 2), True),
                                    StructField("EXT_TIME", StringType(), True),
                                    StructField("SITE_CATEGORY", StringType(), True),
                                    StructField("MRP_TYPE", StringType(), True),
                                    StructField("HELD_STOCK", DoubleType(), True),
                                    StructField("GTIN_NUMBER", StringType(), True),
                                    #StructField("Created_Date", TimestampType(), True)

])

Ingestion raw data into raw table

In [0]:
df = spark.read.format("csv").option("header", "true").option('Inferschema', "true").load(f"{inbound_container}")
dropDuplicates = df.dropDuplicates()

window_spec = Window.orderBy("WIN_NUMBER")

add_missing_columns = dropDuplicates.withColumn('CREATED_DATE', lit(current_timestamp())).withColumn("ID", row_number().over(window_spec))

formatting_date = add_missing_columns.withColumn("REPORT_DATE", to_date("REPORT_DATE", "dd-MM-yyyy"))  

final_df = formatting_date.select("ID","ARTICLE_NUMBER","WIN_NUMBER","SITE","MERCHANDISE_CATEGORY","REPORT_DATE","BASE_UOM","QUANTITY","EXT_TIME","SITE_CATEGORY","MRP_TYPE","HELD_STOCK","GTIN_NUMBER","CREATED_DATE")

final_df.write.format("delta").option("mergeSchema", "true").mode("overwrite").saveAsTable("default.tbl_ioh_I0132_raw")

Staging table

In [0]:
df_stage = spark.sql("select * from default.tbl_ioh_I0132_raw")
df_stage.write.format("delta")..mode("append").saveAsTable("default.tbl_ioh_I0132_stage")



In [0]:
%sql
select * from default.tbl_ioh_i0132_stage

In [0]:
# Clear all cached tables and views
spark.catalog.clearCache()


Processing Curated table

In [0]:
if len(df.columns) > 0:
    group_stage_df.createOrReplaceGlobalTempView("temp_stage")
    merge_stage = spark.sql('''
                            MERGE INTO default.tbl_ioh_I0132_stage t
                            using (
                                SELECT * FROM global_temp.temp_stage where REPORT_DATE = CURRENT_DATE()
                            ) s
                            on t.WIN_NUMBER = s.WIN_NUMBER and t.SITE = s.SITE 
                            when matched then update set 
                                        t.QUANTITY_DAY_10 = t.QUANTITY_DAY_9,
                                        t.QUANTITY_DAY_9 = t.QUANTITY_DAY_8,
                                        t.QUANTITY_DAY_8 = t.QUANTITY_DAY_7,
                                        t.QUANTITY_DAY_7 = t.QUANTITY_DAY_6,
                                        t.QUANTITY_DAY_6 = t.QUANTITY_DAY_5 ,
                                        t.QUANTITY_DAY_5 = t.QUANTITY_DAY_4,
                                        t.QUANTITY_DAY_4 = t.QUANTITY_DAY_3 ,
                                        t.QUANTITY_DAY_3 = t.QUANTITY_DAY_2,
                                        t.QUANTITY_DAY_2 = t.QUANTITY_DAY_1,
                                        t.QUANTITY_DAY_1 = s.QUANTITY_DAY_1,
                                        t.REPORT_DATE = s.REPORT_DATE
                            when not matched then insert *                
                           ''')  
    print("Merge statement has been executed")
else:
    group_stage_df.write.format("delta").mode("append").saveAsTable("default.tbl_ioh_I0132_stage")
    print("INSERTED RECORDS AT FIRST TIME")